# 03_2 · Khung đánh giá và baseline
**Phần 2/4.** Chia dữ liệu theo thời gian (train đến 31/10, validation tháng 11), dựng các metric, rồi đo baseline "bán chạy nhất" trước và sau khi loại mã không phải hàng hóa.

Câu hỏi cần trả lời: baseline cũ có Hit Rate@10 ≈ 0,90 có phải do mã không phải hàng hóa không? Và bao nhiêu phần trăm lần mua trong tháng 11 là mua lại đúng mẫu đã mua?

In [5]:
from recsys_lib import *
from IPython.display import display

inter, item_stats, catalog, n_inv_total = load_cache()
st = load_state(); blacklist = set(st["blacklist"])
print(f"{len(inter):,} dòng | loại {len(blacklist)} mã không phải hàng hóa")

350,663 dòng | loại 35 mã không phải hàng hóa


## 1. Khung chia dữ liệu
Xem phần giải thích lớp `Split` và `evaluate` trong `recsys_lib.py`. Mẫu chỉ được gợi ý nếu có giao dịch trong 60 ngày cuối của train.

## 5. Baseline: bán chạy nhất, và ảnh hưởng của việc loại mã không phải hàng hóa

In [6]:
val_raw  = Split(inter, catalog, set(), *VAL)          # KHÔNG loại mã nào (tái hiện baseline cũ)
val      = Split(inter, catalog, blacklist, *VAL)      # đã loại mã không phải hàng hóa
print("Không lọc :", val_raw.summary()); print("Đã lọc    :", val.summary())

out = []
for name, sp_, by in [("Bán chạy 30 ngày theo SỐ LƯỢNG, không lọc (baseline cũ)", val_raw, "qty"),
                      ("Bán chạy 30 ngày theo SỐ LƯỢNG, đã lọc", val, "qty"),
                      ("Bán chạy 30 ngày theo NGƯỜI MUA, đã lọc", val, "buyers")]:
    m = Popularity(30, by).fit(sp_)
    out.append(summarize(evaluate(sp_, make_scorer(m), exclude_seen=False), name))
base = pd.concat(out)
base[base.nhom.isin(["co_lich_su", "khach_moi"])][["mo_hinh", "nhom", "so_khach", f"recall@{K}", f"ndcg@{K}", f"hit_rate@{K}"]]

Không lọc : train < 01/11/2025: 280,771 dòng, 55,969 khách | mẫu được gợi ý: 1,435 | khách kỳ đánh giá: 9,426 (có lịch sử 3,066, mới 6,360) | trần Recall: 0.904
Đã lọc    : train < 01/11/2025: 209,067 dòng, 55,969 khách | mẫu được gợi ý: 1,428 | khách kỳ đánh giá: 9,426 (có lịch sử 3,066, mới 6,360) | trần Recall: 0.876


,mo_hinh,nhom,so_khach,recall@10,ndcg@10,hit_rate@10
1,"Bán chạy 30 ngày theo SỐ LƯỢNG, không lọc (bas...",co_lich_su,3066,0.2955,0.3278,0.9031
2,"Bán chạy 30 ngày theo SỐ LƯỢNG, không lọc (bas...",khach_moi,6360,0.3095,0.3311,0.8723
1,"Bán chạy 30 ngày theo SỐ LƯỢNG, đã lọc",co_lich_su,3066,0.0444,0.0310,0.1181
2,"Bán chạy 30 ngày theo SỐ LƯỢNG, đã lọc",khach_moi,6360,0.0455,0.0272,0.0992
1,"Bán chạy 30 ngày theo NGƯỜI MUA, đã lọc",co_lich_su,3066,0.0444,0.0310,0.1181
2,"Bán chạy 30 ngày theo NGƯỜI MUA, đã lọc",khach_moi,6360,0.0455,0.0276,0.0992


In [7]:
# Phần trăm lần mua trong kỳ đánh giá là mua lại đúng mẫu đã mua trước đó (quyết định có nên loại mẫu đã mua khỏi gợi ý)
tr_pairs = set(zip(val.train.customer_id, val.train.item))
vw = inter[(inter.ngay >= val.start) & (inter.ngay < val.stop) & ~inter.item.isin(blacklist)]
vw = vw[vw.customer_id.isin(set(val.users))][["customer_id", "item"]].drop_duplicates()
rep = np.mean([(c, i) in tr_pairs for c, i in zip(vw.customer_id, vw.item)])
print(f"Khách có lịch sử: {100*rep:.1f}% cặp (khách, mẫu) trong tháng 11 là mua lại mẫu đã mua trước đó")

Khách có lịch sử: 2.7% cặp (khách, mẫu) trong tháng 11 là mua lại mẫu đã mua trước đó


In [8]:
base.to_csv(REPORTS / "recsys_baseline.csv", index=False)
save_state(rep_pct=float(rep))
print("Đã lưu reports/recsys_baseline.csv")

Đã lưu reports/recsys_baseline.csv


## Việc cần làm trước khi sang phần 3
- So sánh 3 dòng baseline: nếu dòng "đã lọc" thấp hơn rõ rệt dòng "không lọc", nguyên nhân là mã không phải hàng hóa, và đó là một phát hiện đáng đưa vào báo cáo.
- Nếu hai dòng gần như bằng nhau, Hit Rate cao là do bản chất dữ liệu (nhiều khách cùng mua các mẫu bán chạy), không phải lỗi dữ liệu.
- Gửi tôi bảng baseline này nếu muốn tôi giúp viết phần nhận xét.